# Pitch Estimation: Autocorrelation and YIN

Find the period of a voiced frame, decide whether a frame is voiced at all, and compare with librosa's YIN and pYIN.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
from main import (autocorrelation_pitch, constant_pitch, deepest_dip_pitch, evaluate_track, frame_signal,
                  harmonic_signal, intonation_utterance, spectral_peak_pitch, yin)

sample_rate = 16_000
vowel = harmonic_signal(constant_pitch(150.0, sample_rate, 0.3), sample_rate)
frame = frame_signal(vowel, 640, 160)[15]
print(f'loudest peak {spectral_peak_pitch(frame, sample_rate):.1f} Hz, '
      f'autocorrelation {autocorrelation_pitch(vowel, sample_rate)[15]:.1f} Hz, YIN {yin(vowel, sample_rate)[0][15]:.1f} Hz')

The loudest peak is a harmonic under the first formant. Change the pitch to 120 Hz and predict the new loudest peak before running the cell again: which multiple of 120 Hz sits nearest 730 Hz?

In [ ]:
for f0 in (120.0, 200.0, 300.0):
    voice = harmonic_signal(constant_pitch(f0, sample_rate, 0.3), sample_rate)
    print(f'{f0:5.0f} Hz voice: deepest dip {np.median(deepest_dip_pitch(voice, sample_rate)[5:-5]):6.1f} Hz, '
          f'YIN first dip {np.nanmedian(yin(voice, sample_rate)[0][5:-5]):6.1f} Hz')

In [ ]:
import librosa

utterance, truth = intonation_utterance(sample_rate)
ours_f0, ours_voiced, _ = yin(utterance, sample_rate)
librosa_f0 = librosa.yin(utterance, fmin=60, fmax=400, sr=sample_rate, frame_length=640, hop_length=160)
pyin_f0, pyin_voiced, _ = librosa.pyin(utterance, fmin=60, fmax=400, sr=sample_rate, frame_length=640, hop_length=160)

for name, f0, voiced in [('ours', ours_f0, ours_voiced),
                         ('librosa.yin', librosa_f0, np.ones(len(librosa_f0), dtype=bool)),
                         ('librosa.pyin', pyin_f0, pyin_voiced)]:
    scores = evaluate_track(f0, voiced, truth, 640, 160)
    print(f"{name:<13} voicing {scores['voicing_accuracy']:.1%}  GPE {scores['gross_pitch_error']:.1%}  "
          f"median {scores['median_cents']:.1f} cents")

`librosa.yin` has no voicing output, so it reports a pitch even during the hiss. Now pass `frame_length=1_024` to our `yin` and to `evaluate_track`. Why does voicing accuracy collapse on the glides while it stays perfect on a steady vowel?